# View 2: Latent Space Projections (UMAP)
Visualizes AkidaNet penultimate representation drift under corruption and where the CNN and Akida model disagree.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap

In [ ]:
df = pd.read_parquet("../results/akidanet_imagenetc.parquet")
os.makedirs("../figures/phase1_human", exist_ok=True)

features = np.vstack(df["embedding"].values)

reducer = umap.UMAP(n_neighbors=25, min_dist=0.15, random_state=0)
embeddings_2d = reducer.fit_transform(features)

df["umap_x"] = embeddings_2d[:, 0]
df["umap_y"] = embeddings_2d[:, 1]
df["agreement"] = np.where(df["cnn_pred"] == df["akida_pred"], "agree", "disagree")

Plot Severity 1 vs Severity 5 Latent Drift

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8), sharex=True, sharey=True)

for ax, sev in zip(axes, [1, 5]):
    df_s = df[df["severity"] == sev]
    sns.scatterplot(
        data=df_s,
        x="umap_x",
        y="umap_y",
        hue="corruption",
        hue_order=["fog", "gaussian_noise", "contrast"],
        palette={"fog": "#1b9e77", "gaussian_noise": "#d95f02", "contrast": "#e7298a"},
        s=8,
        alpha=0.6,
        linewidth=0,
        ax=ax
    )
    disagree = df_s[df_s["agreement"] == "disagree"]
    ax.scatter(disagree["umap_x"], disagree["umap_y"], s=6, c="black", linewidths=0,
               label=f"CNN/Akida disagree ({len(disagree) / len(df_s):.0%})")
    leg = ax.legend(loc="upper left", markerscale=2)
    for h in leg.legend_handles:
        h.set_alpha(1)
    ax.set_title(f"UMAP Latent Space: Severity {sev} ({df['akida_source'].iloc[0]})", fontsize=14)

plt.tight_layout()
plt.savefig("../figures/phase1_human/umap_representation_drift.png", dpi=300)
plt.show()
